In [ ]:
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess,Fourier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_squared_error

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, ElasticNet, Lasso, Ridge
from sklearn.neural_network import MLPRegressor

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit

from sklearn.preprocessing import StandardScaler, MinMaxScaler

import itertools

import warnings



In [3]:
from utils.common import import_dataframe

In [4]:
# Ignore warnings that match the specified criteria
warnings.filterwarnings('ignore', message='.*deprecated.*', category=DeprecationWarning)
# Issue a warning
warnings.warn('This is a deprecated feature', DeprecationWarning)

nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
nome_coluna_vazao_jusante = "Vazão Jusante (m³/s)"

df = import_dataframe()
df.set_index("Data", inplace=True)


In [ ]:

# --- CONFIGURAÇÃO GLOBAL ---
VALIDATION_SIZE = 90
N_SPLITS = 5
COL_VOL = "Volume Útil Armazenado (%)"
COL_VN = "Vazão Natural (m³/s)"
LAGS_CICLO = 2
LAGS_VOLUME = 1

# =============================================================================
# 1. FUNÇÕES AUXILIARES
# =============================================================================
def calcular_climatologia_robusta(df_train, col_target, window_days=7):
    temp = df_train[[col_target]].copy()
    temp['day_of_year'] = temp.index.dayofyear
    climatologia = {}
    for day in range(1, 367):
        d_min = day - (window_days // 2)
        d_max = day + (window_days // 2)
        mask = (temp['day_of_year'] >= d_min) & (temp['day_of_year'] <= d_max)
        vals = temp.loc[mask, col_target]
        if len(vals) > 0:
            vals = vals[(vals >= vals.quantile(0.10)) & (vals <= vals.quantile(0.90))]
            media = vals.mean() if len(vals) > 0 else 0
        else:
            media = 0
        climatologia[day] = media
    return pd.Series(climatologia, name='clim_base')

def get_clim_feature_array(index, curva_clim):
    days = index.dayofyear
    return np.array([curva_clim.get(d, curva_clim.iloc[-1]) for d in days])

def criar_feat_fourier(index):
    # Cria features determinísticas (Tendência + Sazonalidade Anual)
    dp = DeterministicProcess(
        index=index, constant=True, order=1, seasonal=False, 
        additional_terms=[CalendarFourier("YE", 2)], drop=True
    )
    return dp.in_sample()

def criar_lags(series, lags):
    df = pd.DataFrame(index=series.index)
    for l in range(1, lags + 1):
        df[f'lag_{l}'] = series.shift(l)
    return df

# =============================================================================
# 2. CLASSE GENERALISTA (COM SUPORTE A VOLUME HÍBRIDO)
# =============================================================================
class ReservoirForecaster:
    def __init__(self, config):
        self.cfg = config
        self.models = {}
        self.scalers = {}
        self.history = {} 
        self.vol_features_ = [] 

    def _train_flow(self, df_train):
        y = df_train[COL_VN]
        
        # 1. Climatologia (Base)
        clim_curve = calcular_climatologia_robusta(df_train, COL_VN)
        self.history['flow_clim'] = clim_curve
        
        # 2. Estratégia: Extração de Resíduo via Fourier
        X_math = criar_feat_fourier(df_train.index)
        model_math = LinearRegression()
        model_math.fit(X_math, y)
        
        y_math_clean = pd.Series(model_math.predict(X_math), index=y.index)
        residuo = y - y_math_clean
        
        # 3. Treina ML no Resíduo (Ciclo)
        X_lags = criar_lags(residuo, LAGS_CICLO).dropna()
        y_target = residuo.loc[X_lags.index]
        
        model_cycle = self.cfg['model_flow_cycle']
        model_cycle.fit(X_lags, y_target)
        self.models['flow_cycle'] = model_cycle
        
        return residuo.tail(LAGS_CICLO).values

    def _train_volume(self, df_train):
        y = df_train[COL_VOL]
        X = pd.DataFrame(index=df_train.index)
        
        # --- Feature Engineering Dinâmica ---
        X['vol_lag1'] = df_train[COL_VOL].shift(1)
        
        if self.cfg.get('use_diff1', True):
            X['vol_diff'] = df_train[COL_VOL].shift(1) - df_train[COL_VOL].shift(2)
            
        if self.cfg.get('use_diff2', False):
            X['vol_diff2'] = (df_train[COL_VOL].shift(1) - df_train[COL_VOL].shift(2)) - \
                             (df_train[COL_VOL].shift(2) - df_train[COL_VOL].shift(3))
        
        # Input Exógeno: Vazão
        X['vazao_input'] = df_train[COL_VN]
        
        # Input Exógeno: Climatologia Volume
        if self.cfg.get('use_vol_clim', False):
            clim_vol = calcular_climatologia_robusta(df_train, COL_VOL)
            self.history['vol_clim'] = clim_vol
            X['vol_clim'] = get_clim_feature_array(df_train.index, clim_vol)

        # Input Exógeno: Tendência Híbrida (Fourier no Volume) -> IMPLEMENTADO AQUI
        if self.cfg.get('vol_hybrid_trend', False):
            # Cria Fourier para o índice de treino
            X_fourier = criar_feat_fourier(df_train.index)
            # Concatena (Pandas alinha pelo índice)
            X = pd.concat([X, X_fourier], axis=1)

        X_full = X.dropna()
        y_train = y.loc[X_full.index]
        
        # Salva features para garantir ordem no predict
        self.vol_features_ = X_full.columns.tolist()
        
        # Scaling
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_full)
        self.scalers['vol'] = scaler
        
        # Treino
        model = self.cfg['model_volume']
        model.fit(X_scaled, y_train)
        self.models['volume'] = model

    def predict(self, df_test, buffer_vol, last_resid_flow):
        preds_vol = []
        
        curr_buffer_vol = list(buffer_vol)
        curr_resid_flow = list(last_resid_flow)
        
        # --- PREPARAÇÃO TREND HÍBRIDA (SE NECESSÁRIO) ---
        # Gera o Fourier para todo o período de teste de uma vez (mais eficiente)
        X_trend_test = None
        if self.cfg.get('vol_hybrid_trend', False):
            X_trend_test = criar_feat_fourier(df_test.index)
        
        for i in range(len(df_test)):
            date = df_test.index[i]
            
            # --- 1. VAZÃO (Híbrida: Base Clim + Ciclo ML) ---
            val_base_clim = get_clim_feature_array(pd.DatetimeIndex([date]), self.history['flow_clim'])[0]
            
            lags_cycle = np.array(curr_resid_flow[-LAGS_CICLO:][::-1])
            cycle_pred = self.models['flow_cycle'].predict(pd.DataFrame([lags_cycle], columns=[f'lag_{k}' for k in range(1, LAGS_CICLO+1)]))[0]
            
            # Damping Logic
            if self.cfg.get('cycle_logic') == 'damping_specific':
                if cycle_pred > 0:
                    last_r = curr_resid_flow[-1]
                    if cycle_pred - last_r > 0:
                        cycle_pred *= 0.2
                    else:
                        cycle_pred *= 0.8
            
            vazao_final = max(0, val_base_clim + cycle_pred)
            curr_resid_flow.append(cycle_pred)
            
            # --- 2. VOLUME ---
            vol_t1 = curr_buffer_vol[-1]
            vol_t2 = curr_buffer_vol[-2]
            vol_t3 = curr_buffer_vol[-3]
            
            # Monta features base (Dicionário não garante ordem, DF garante)
            feat_dict = {'vol_lag1': vol_t1}
            
            if self.cfg.get('use_diff1', True):
                feat_dict['vol_diff'] = vol_t1 - vol_t2
                
            if self.cfg.get('use_diff2', False):
                feat_dict['vol_diff2'] = (vol_t1 - vol_t2) - (vol_t2 - vol_t3)
                
            feat_dict['vazao_input'] = vazao_final
            
            if self.cfg.get('use_vol_clim', False):
                feat_dict['vol_clim'] = get_clim_feature_array(pd.DatetimeIndex([date]), self.history['vol_clim'])[0]
            
            # Cria DataFrame parcial
            input_vol = pd.DataFrame([feat_dict])
            
            # Adiciona Trend Híbrida se necessário
            if self.cfg.get('vol_hybrid_trend', False):
                # Pega a linha correspondente do Fourier pré-calculado
                # Reset index para concatenar lateralmente sem problemas de índice (input_vol tem index 0)
                row_trend = X_trend_test.iloc[[i]].reset_index(drop=True)
                input_vol = pd.concat([input_vol, row_trend], axis=1)
            
            # REORDENAÇÃO CRÍTICA: Garante que as colunas estão na mesma ordem do fit
            # Isso corrige qualquer erro de concatenação ou ordem de dicionário
            input_vol = input_vol[self.vol_features_]
            
            # Scaling
            input_vol_scaled = self.scalers['vol'].transform(input_vol)
            
            # Predict
            pred_vol = self.models['volume'].predict(input_vol_scaled)[0]
            preds_vol.append(pred_vol)
            
            curr_buffer_vol.append(pred_vol)
            curr_buffer_vol.pop(0)
            
        return preds_vol

    def evaluate(self, df):
        tscv = TimeSeriesSplit(n_splits=N_SPLITS, test_size=VALIDATION_SIZE)
        rmse_scores = []
        mae_scores = []
        
        for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
            df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]
            
            last_resid_flow = self._train_flow(df_train)
            self._train_volume(df_train)
            
            buffer_vol = df_train[COL_VOL].tail(3).values
            preds_vol = self.predict(df_test, buffer_vol, last_resid_flow)
            
            rmse = np.sqrt(mean_squared_error(df_test[COL_VOL], preds_vol))
            rmse_scores.append(rmse)
            mae = mean_absolute_error(df_test[COL_VOL], preds_vol)
            mae_scores.append(mae)
        
        return np.mean(rmse_scores), np.mean(mae_scores)

# =============================================================================
# 3. EXECUÇÃO DO GRID SEARCH
# =============================================================================

param_grid = {
    # Modelo Vazão (Fixo na sua melhor configuração)
    'model_flow_cycle': [KNeighborsRegressor(n_neighbors=5), RandomForestRegressor(max_depth=3), RandomForestRegressor(max_depth=5)],
    'cycle_logic': ['damping_specific', None], 
    
    # Modelo Volume (Variações)
    'model_volume': [LinearRegression(), Ridge(alpha=1.0)],
    'use_diff1': [True, False],
    'use_diff2': [True, False],      
    'use_vol_clim': [True, False],   
    
    # NOVA FEATURE PARA TESTE
    'vol_hybrid_trend': [True, False] # Testa se incluir Fourier direto no volume ajuda
}

keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Total de configurações a testar: {len(combinations)}")
print("=== INICIANDO GRID SEARCH ===")

results = []

for i, config in enumerate(combinations):
    try:
        cfg_run = config.copy()
        forecaster = ReservoirForecaster(cfg_run)
        rmse, mae = forecaster.evaluate(df)
        
        res = config.copy()
        res['rmse'] = rmse
        res['mae'] = mae
        # Converter objetos para string para o DataFrame ficar legível
        res['model_volume'] = str(config['model_volume'])
        res['model_flow_cycle'] = str(config['model_flow_cycle'])
        res['cycle_logic'] = str(config['cycle_logic'])
        res['use_diff1'] = str(config['use_diff1'])
        res['use_diff2'] = str(config['use_diff2'])
        res['use_vol_clim'] = str(config['use_vol_clim'])
        res['vol_hybrid_trend'] = str(config['vol_hybrid_trend'])
        
        results.append(res)
        print(f"Iter {i+1}: RMSE={rmse:.4f}, MAE = {mae:.4} | Diff2={config['use_diff2']} | VolClim={config['use_vol_clim']} | HybridTrend={config['vol_hybrid_trend']}")
        
    except Exception as e:
        print(f"Erro na config {i}: {e}")
        # raise e # Descomente para debugar erro completo

# --- ANÁLISE ---
df_results = pd.DataFrame(results).sort_values(by='rmse')
print("\n=== TOP 5 MELHORES CONFIGURAÇÕES ===")
print(df_results.head(5))

Total de configurações a testar: 192
=== INICIANDO GRID SEARCH ===
Iter 1: RMSE=3.5178, MAE = 2.966 | Diff2=True | VolClim=True | HybridTrend=True
Iter 2: RMSE=2.5276, MAE = 2.187 | Diff2=True | VolClim=True | HybridTrend=False
Iter 3: RMSE=3.5035, MAE = 2.951 | Diff2=True | VolClim=False | HybridTrend=True
Iter 4: RMSE=2.1211, MAE = 1.817 | Diff2=True | VolClim=False | HybridTrend=False
Iter 5: RMSE=3.4265, MAE = 2.891 | Diff2=False | VolClim=True | HybridTrend=True
Iter 6: RMSE=2.5667, MAE = 2.232 | Diff2=False | VolClim=True | HybridTrend=False
Iter 7: RMSE=3.4093, MAE = 2.873 | Diff2=False | VolClim=False | HybridTrend=True
